# NOVA REAL VIDEO / FREE v1

Цель: настоящий генеративный 5-секундный клип, а не 2D warp. Платные API запрещены. Skeleton используется только как motion/control reference.


In [ ]:
PAID_APIS_ALLOWED = False
DURATION = 5
TARGET_PROFILE = 'Wan 2.2 TextImage2Video 5B FastWan'
TARGET_RESOLUTION = '480p'
PROMPT = '''Two adults from the reference image walk naturally side by side toward the camera on a rainy neon city street. Full believable walking cycles with heel-to-toe contact, planted stance feet, natural knee flexion, clear pelvis weight transfer, relaxed arms swinging in opposite phase to the legs, subtle shoulder bounce and chest movement from genuine laughter. Preserve the exact identities, faces, hairstyle, beard, clothing, jacket structure, proportions and environment from the reference image. Clothing remains structurally stable and follows the body naturally without stretching or breathing. Smooth stabilized camera tracking backward. Photorealistic live-action motion, coherent anatomy, coherent lighting, temporal consistency, no overlays.'''
NEGATIVE = '''rubber body, elastic body, jelly motion, breathing clothes, fabric swimming, cloth warping, texture swimming, melted jacket, changing clothes, identity drift, face morphing, facial distortion, mouth distortion, changing hair, changing beard, extra limbs, extra arms, extra legs, extra fingers, broken anatomy, twisted joints, foot sliding, moonwalk, floating feet, hovering feet, duplicated person, fused bodies, camera jitter, flicker, temporal instability, warped background, geometry wobble, body stretching, unnatural shoulder motion, exaggerated bounce, cartoon motion, HUD, text, subtitles, watermark'''
assert PAID_APIS_ALLOWED is False
print('FREE MODE LOCKED')
print('Model profile:', TARGET_PROFILE)
print('Resolution:', TARGET_RESOLUTION)
print('Duration:', DURATION)


## Запуск WanGP на бесплатном Colab GPU
Free T4 обычно имеет около 15 GB VRAM, поэтому используем 5B FastWan и 480p. Более тяжёлые модели не запускаются автоматически.


In [ ]:
import json, subprocess, sys
from pathlib import Path
subprocess.run(['nvidia-smi'], check=True)
ROOT = Path('/content/Wan2GP-on-Colab')
PIN = 'e428b5ebc0d49589474ef5d81e05cc2ab3c1e17b'
if not (ROOT/'.git').exists():
    subprocess.run(['git','clone','https://github.com/Square-Zero-Labs/Wan2GP-on-Colab.git',str(ROOT)],check=True)
subprocess.run(['git','-C',str(ROOT),'fetch','--depth','1','origin',PIN],check=True)
subprocess.run(['git','-C',str(ROOT),'checkout','--detach',PIN],check=True)
nb=json.loads((ROOT/'wan2gp-google-colab.ipynb').read_text())
launch=None
for i,c in enumerate(nb['cells']):
    if c.get('cell_type')!='code': continue
    code=''.join(c.get('source') or [])
    if 'Launching Wan2GP' in code:
        launch=code; continue
    code=code.replace('USE_GOOGLE_DRIVE_DATA = False','USE_GOOGLE_DRIVE_DATA = False')
    exec(compile(code,f'wan_colab_cell_{i}','exec'),globals())
if launch is None: raise RuntimeError('WanGP launch cell not found')
print('WanGP installed. Use only the free profile printed above.')


## Quality rules
1. Reference image = identity/wardrobe/world.
2. Skeleton/control = motion only. Never composite the visible skeleton into the final.
3. Keep 5 seconds for the first run.
4. Use 480p generation on free T4, then upscale after generation.
5. Reject output if feet slide, clothing swims, faces drift, anatomy breaks, or camera flickers.
6. Never fall back to 2D warp as a final video.


In [ ]:
print('PROMPT:
', PROMPT)
print('
NEGATIVE:
', NEGATIVE)
exec(compile(launch,'verified_wangp_launch','exec'),globals())
